## Лабораторная работа №4
### Линейные модели, SVM и деревья решений

**Цель работы** – изучить применение:
- логистической регрессии,
- метода опорных векторов (SVM),
- дерева решений.

Для решения задачи многоклассовой классификации на примере датасета **Iris** (3 вида ирисов).

**Используемые библиотеки:**  
Microsoft.Data.Analysis, Plotly.NET.CSharp, Microsoft.ML, MathNet.Numerics.

## 1. Датасет Iris

Признаки (все числовые):
- `SepalLength` – длина чашелистика
- `SepalWidth`  – ширина чашелистика
- `PetalLength` – длина лепестка
- `PetalWidth`  – ширина лепестка

Целевая переменная `Species` – вид ириса:
- `Iris-setosa`
- `Iris-versicolor`
- `Iris-virginica`

Пропусков нет, категориальные признаки отсутствуют.  
Выполним только **масштабирование** признаков.

In [30]:
#r "nuget: Microsoft.Data.Analysis, 0.22.1"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "nuget: Plotly.NET.CSharp, 0.12.0"
#r "nuget: MathNet.Numerics, 5.0.0"

Installed Packages MathNet.Numerics, 5.0.0 Microsoft.Data.Analysis, 0.22.1 Plotly.NET.CSharp, 0.12.0 Plotly.NET.Interactive, 5.0.0

In [31]:
using Plotly.NET;
using PlotlyCS = Plotly.NET.CSharp.Chart;
using Microsoft.Data.Analysis;
using MathNet.Numerics.Statistics;
using System.Linq;
using System.IO;
using System.Net;
using System.Globalization;

In [32]:
var dataDir = Path.Combine(Directory.GetCurrentDirectory(), "Data");
Directory.CreateDirectory(dataDir);
var dataPath = Path.Combine(dataDir, "iris.csv");

if (!File.Exists(dataPath))
{
    var url = "https://raw.githubusercontent.com/uiuc-cse/data-fa14/gh-pages/data/iris.csv";
    Console.WriteLine("Скачиваю iris.csv...");
    new WebClient().DownloadFile(url, dataPath);
    Console.WriteLine("Готово.");
}

string[] colNames = { "SepalLength", "SepalWidth", "PetalLength", "PetalWidth", "Species" };
Type[] colTypes = { typeof(float), typeof(float), typeof(float), typeof(float), typeof(string) };

var df = DataFrame.LoadCsv(dataPath,
    separator: ',',
    header: true,
    columnNames: colNames,
    dataTypes: colTypes,
    cultureInfo: CultureInfo.InvariantCulture);

df.Head(3).Display();

index,SepalLength,SepalWidth,PetalLength,PetalWidth,Species
0,4.9,3,1.4,0.2,Iris-setosa
1,4.7,3.2,1.3,0.2,Iris-setosa
2,4.6,3.1,1.5,0.2,Iris-setosa



(9,5): warning SYSLIB0014: "WebClient.WebClient()" является устаревшим: 'WebRequest, HttpWebRequest, ServicePoint, and WebClient are obsolete. Use HttpClient instead.' (https://aka.ms/dotnet-warnings/SYSLIB0014)



## 2. Визуализация исходных данных

Покажем распределение признаков и соотношение классов.

In [33]:
// Гистограмма одного признака (пример SepalLength)
var hist = PlotlyCS.Histogram<float, float, string>(X: df["SepalLength"].Cast<float>().ToArray())
    .WithTitle("Распределение длины чашелистика");
display(hist);

// Круговая диаграмма видов
var speciesCounts = df["Species"].Cast<string>()
    .GroupBy(s => s)
    .ToDictionary(g => g.Key, g => g.Count());
var pie = PlotlyCS.Pie<int, string, string>(
    values: speciesCounts.Values.ToArray(),
    Labels: speciesCounts.Keys.ToArray())
    .WithTitle("Соотношение видов ирисов");
display(pie);

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

## 3. Подготовка данных и масштабирование

In [34]:
// Label Encoding для видов
var speciesCol = df["Species"];
var labelMapping = new Dictionary<string, int>
{
    {"Iris-setosa", 0},
    {"Iris-versicolor", 1},
    {"Iris-virginica", 2}
};
var labels = new int[df.Rows.Count];
for (long i = 0; i < df.Rows.Count; i++)
    labels[i] = labelMapping[(string)speciesCol[i]];

df.Columns.Remove("Species");
df.Columns.Add(new PrimitiveDataFrameColumn<int>("Label", labels));

Console.WriteLine($"Примеры меток: {string.Join(", ", labels.Take(10))}");

Примеры меток: 0, 0, 0, 0, 0, 0, 0, 0, 0, 0


In [35]:
// Функция масштабирования float-столбца
void Standardize(DataFrame df, string colName)
{
    var col = df[colName];
    var vals = new List<float>();
    for (long i = 0; i < df.Rows.Count; i++)
        if (col[i] is float f) vals.Add(f);

    double mean = vals.Average();
    double std = Math.Sqrt(vals.Average(v => (v - mean) * (v - mean)));
    for (long i = 0; i < df.Rows.Count; i++)
        if (col[i] is float f)
            col[i] = (float)((f - mean) / std);
}

string[] featureNames = { "SepalLength", "SepalWidth", "PetalLength", "PetalWidth" };
foreach (var name in featureNames)
    Standardize(df, name);

Console.WriteLine("Признаки масштабированы (StandardScaler).");

Признаки масштабированы (StandardScaler).


In [36]:
int nRows = (int)df.Rows.Count;
double[][] X = new double[nRows][];
int[] y = new int[nRows];

for (int i = 0; i < nRows; i++)
{
    X[i] = new double[4];
    for (int j = 0; j < 4; j++)
        X[i][j] = (float)df[featureNames[j]][i];
    y[i] = (int)df["Label"][i];
}

// train_test_split (80/20)
(int[] trainIdx, int[] testIdx) = TrainTestSplit(Enumerable.Range(0, nRows).ToArray(), 0.2, 42);
double[][] X_train = trainIdx.Select(i => X[i]).ToArray();
int[] y_train = trainIdx.Select(i => y[i]).ToArray();
double[][] X_test = testIdx.Select(i => X[i]).ToArray();
int[] y_test = testIdx.Select(i => y[i]).ToArray();

Console.WriteLine($"Train: {X_train.Length} объектов, Test: {X_test.Length} объектов");

// --- Вспомогательная функция ---
(int[] train, int[] test) TrainTestSplit(int[] indices, double testSize, int seed)
{
    var rnd = new Random(seed);
    var shuffled = indices.OrderBy(i => rnd.Next()).ToArray();
    int testCount = (int)(shuffled.Length * testSize);
    return (shuffled.Skip(testCount).ToArray(), shuffled.Take(testCount).ToArray());
}

Train: 120 объектов, Test: 29 объектов


## 4. Логистическая регрессия (ML.NET)

In [37]:
// Вспомогательные функции для оценки качества
double Accuracy(int[] yTrue, int[] yPred)
{
    int correct = 0;
    for (int i = 0; i < yTrue.Length; i++)
        if (yTrue[i] == yPred[i]) correct++;
    return (double)correct / yTrue.Length;
}

double MacroF1(int[] yTrue, int[] yPred)
{
    var classes = yTrue.Distinct().OrderBy(c => c).ToArray();
    double sumF1 = 0;
    foreach (var cls in classes)
    {
        int tp = yTrue.Zip(yPred, (a, b) => a == cls && b == cls).Count(v => v);
        int fp = yPred.Count(p => p == cls) - tp;
        int fn = yTrue.Count(a => a == cls) - tp;
        double prec = tp + fp > 0 ? (double)tp / (tp + fp) : 0;
        double rec = tp + fn > 0 ? (double)tp / (tp + fn) : 0;
        double f1 = prec + rec > 0 ? 2 * prec * rec / (prec + rec) : 0;
        sumF1 += f1;
    }
    return sumF1 / classes.Length;
}

In [39]:
// One-hot encoding для меток (3 класса)
int K = 3; // число классов
double[][] y_train_ohe = y_train.Select(l =>
{
    var arr = new double[K];
    arr[l] = 1.0;
    return arr;
}).ToArray();

double[][] y_test_ohe = y_test.Select(l =>
{
    var arr = new double[K];
    arr[l] = 1.0;
    return arr;
}).ToArray();

// Инициализация весов и смещений
int D = X_train[0].Length; // размерность признаков
var rnd = new Random(42);
double[][] W = new double[K][];
for (int k = 0; k < K; k++)
{
    W[k] = new double[D];
    for (int d = 0; d < D; d++)
        W[k][d] = rnd.NextDouble() * 0.01;
}
double[] b = new double[K]; // смещения, изначально 0

// Softmax
double[] Softmax(double[] z)
{
    double max = z.Max();
    double[] exp = z.Select(zi => Math.Exp(zi - max)).ToArray();
    double sum = exp.Sum();
    return exp.Select(e => e / sum).ToArray();
}

// Градиентный спуск
double lr = 0.1;
int epochs = 1000;
int N_train = X_train.Length;

for (int epoch = 0; epoch < epochs; epoch++)
{
    // Перемешивание данных
    var indices = Enumerable.Range(0, N_train).OrderBy(i => rnd.Next()).ToArray();
    foreach (int i in indices)
    {
        double[] x = X_train[i];
        double[] y_true = y_train_ohe[i];
        // Прямой проход
        double[] z = new double[K];
        for (int k = 0; k < K; k++)
        {
            double s = b[k];
            for (int d = 0; d < D; d++)
                s += W[k][d] * x[d];
            z[k] = s;
        }
        double[] probs = Softmax(z);
        // Обновление весов (градиент)
        for (int k = 0; k < K; k++)
        {
            double error = y_true[k] - probs[k];
            for (int d = 0; d < D; d++)
                W[k][d] += lr * error * x[d];
            b[k] += lr * error;
        }
    }
}

// Предсказание
int[] PredictLr(double[][] X)
{
    int[] preds = new int[X.Length];
    for (int i = 0; i < X.Length; i++)
    {
        double[] z = new double[K];
        for (int k = 0; k < K; k++)
        {
            double s = b[k];
            for (int d = 0; d < D; d++)
                s += W[k][d] * X[i][d];
            z[k] = s;
        }
        preds[i] = Array.IndexOf(Softmax(z), Softmax(z).Max());
    }
    return preds;
}

int[] predLr = PredictLr(X_test);
double accLr = Accuracy(y_test, predLr);
double f1Lr = MacroF1(y_test, predLr);
Console.WriteLine($"Logistic Regression (Softmax) -> Accuracy: {accLr:F3}, Macro-F1: {f1Lr:F3}");

Logistic Regression (Softmax) -> Accuracy: 0,931, Macro-F1: 0,929


## 5. SVM (ML.NET)

In [45]:
int[] predSvm = new int[X_test.Length];
double[][] W_svm = new double[K][];
double[] b_svm = new double[K];

// Обучаем по одному классификатору на класс (One-vs-Rest)
for (int class_k = 0; class_k < K; class_k++)
{
    // Бинарные метки: +1 если класс k, -1 иначе
    int[] y_bin = y_train.Select(v => v == class_k ? 1 : -1).ToArray();
    
    double[] w = new double[D];
    double bias = 0.0;
    double lambda = 0.01; // регуляризация
    double lr_svm = 0.01;
    
    for (int epoch = 0; epoch < 500; epoch++)
    {
        var indices = Enumerable.Range(0, N_train).OrderBy(i => rnd.Next()).ToArray();
        foreach (int i in indices)
        {
            double[] x = X_train[i];
            int yi = y_bin[i];
            double decision = bias;
            for (int d = 0; d < D; d++)
                decision += w[d] * x[d];
            if (yi * decision < 1) // нарушение отступа
            {
                for (int d = 0; d < D; d++)
                    w[d] += lr_svm * (yi * x[d] - 2 * lambda * w[d]);
                bias += lr_svm * yi;
            }
            else
            {
                for (int d = 0; d < D; d++)
                    w[d] -= lr_svm * 2 * lambda * w[d];
            }
        }
    }
    W_svm[class_k] = w;
    b_svm[class_k] = bias;
}

// Предсказание: выбираем класс с наибольшим значением w*x + b
for (int i = 0; i < X_test.Length; i++)
{
    double[] scores = new double[K];
    for (int k = 0; k < K; k++)
    {
        double s = b_svm[k];
        for (int d = 0; d < D; d++)
            s += W_svm[k][d] * X_test[i][d];
        scores[k] = s;
    }
    predSvm[i] = Array.IndexOf(scores, scores.Max());
}

double accSvm = Accuracy(y_test, predSvm);
double f1Svm = MacroF1(y_test, predSvm);
Console.WriteLine($"SVM (Linear, OvR) -> Accuracy: {accSvm:F3}, Macro-F1: {f1Svm:F3}");

SVM (Linear, OvR) -> Accuracy: 0,931, Macro-F1: 0,935


## 6. Дерево решений (собственная реализация)

Реализуем простой классификатор: разбиение по энтропии, лист – большинство классов.  
Дерево будет маленьким, его легко визуализировать текстом.

In [46]:
public class DecisionTree
{
    public static string[] FeatureNames;   // для доступа внутри методов

    private Node _root;
    private int _maxDepth;

    public DecisionTree(int maxDepth = 5)
    {
        _maxDepth = maxDepth;
    }

    public void Fit(double[][] X, int[] y)
    {
        _root = BuildTree(X, y, 0);
    }

    public int[] Predict(double[][] X)
    {
        return X.Select(x => PredictSingle(_root, x)).ToArray();
    }

    private int PredictSingle(Node node, double[] x)
    {
        if (node.IsLeaf)
            return node.PredictedClass;
        if (x[node.FeatureIndex] <= node.Threshold)
            return PredictSingle(node.Left, x);
        else
            return PredictSingle(node.Right, x);
    }

    private Node BuildTree(double[][] X, int[] y, int depth)
    {
        // Критерии остановки
        if (y.Distinct().Count() == 1 || depth >= _maxDepth)
            return new Node { IsLeaf = true, PredictedClass = y.GroupBy(v => v).OrderByDescending(g => g.Count()).First().Key };

        int bestFeature = 0;
        double bestThreshold = 0;
        double bestGain = 0;
        bool found = false;

        for (int f = 0; f < X[0].Length; f++)
        {
            var values = X.Select(row => row[f]).Distinct().OrderBy(v => v).ToArray();
            for (int i = 0; i < values.Length - 1; i++)
            {
                double threshold = (values[i] + values[i + 1]) / 2.0;
                var left = X.Select((row, idx) => (row, idx)).Where(t => t.row[f] <= threshold).Select(t => t.idx).ToArray();
                var right = X.Select((row, idx) => (row, idx)).Where(t => t.row[f] > threshold).Select(t => t.idx).ToArray();
                if (left.Length == 0 || right.Length == 0) continue;

                double gain = InfoGain(y, left, right);
                if (gain > bestGain)
                {
                    bestGain = gain;
                    bestFeature = f;
                    bestThreshold = threshold;
                    found = true;
                }
            }
        }

        if (!found)
            return new Node { IsLeaf = true, PredictedClass = y.GroupBy(v => v).OrderByDescending(g => g.Count()).First().Key };

        var leftIdx = X.Select((row, idx) => (row, idx)).Where(t => t.row[bestFeature] <= bestThreshold).Select(t => t.idx).ToArray();
        var rightIdx = X.Select((row, idx) => (row, idx)).Where(t => t.row[bestFeature] > bestThreshold).Select(t => t.idx).ToArray();

        return new Node
        {
            FeatureIndex = bestFeature,
            Threshold = bestThreshold,
            Left = BuildTree(leftIdx.Select(i => X[i]).ToArray(), leftIdx.Select(i => y[i]).ToArray(), depth + 1),
            Right = BuildTree(rightIdx.Select(i => X[i]).ToArray(), rightIdx.Select(i => y[i]).ToArray(), depth + 1)
        };
    }

    private double InfoGain(int[] y, int[] left, int[] right)
    {
        double parentEntropy = Entropy(y);
        double leftEntropy = Entropy(left.Select(i => y[i]));
        double rightEntropy = Entropy(right.Select(i => y[i]));
        double p = (double)left.Length / y.Length;
        return parentEntropy - p * leftEntropy - (1 - p) * rightEntropy;
    }

    private double Entropy(IEnumerable<int> labels)
    {
        var groups = labels.GroupBy(v => v).Select(g => g.Count()).ToArray();
        double total = groups.Sum();
        if (total == 0) return 0;
        return -groups.Sum(c => (c / total) * Math.Log(c / total, 2));
    }

    public void PrintTree()
    {
        PrintNode(_root, "");
    }

    private void PrintNode(Node node, string indent)
    {
        if (node.IsLeaf)
        {
            Console.WriteLine($"{indent}➤ класс {node.PredictedClass}");
        }
        else
        {
            Console.WriteLine($"{indent}[{FeatureNames[node.FeatureIndex]} <= {node.Threshold:F2}]");
            Console.WriteLine($"{indent}├─ Да:");
            PrintNode(node.Left, indent + "│  ");
            Console.WriteLine($"{indent}└─ Нет:");
            PrintNode(node.Right, indent + "   ");
        }
    }

    public Dictionary<string, int> FeatureImportance()
    {
        var importance = new Dictionary<string, int>();
        foreach (var name in FeatureNames)
            importance[name] = 0;
        CollectImportance(_root, importance);
        return importance;
    }

    private void CollectImportance(Node node, Dictionary<string, int> imp)
    {
        if (node.IsLeaf) return;
        imp[FeatureNames[node.FeatureIndex]]++;
        CollectImportance(node.Left, imp);
        CollectImportance(node.Right, imp);
    }

    private class Node
    {
        public bool IsLeaf;
        public int PredictedClass;
        public int FeatureIndex;
        public double Threshold;
        public Node Left;
        public Node Right;
    }
}


(3,28): warning CS0649: Полю "DecisionTree.FeatureNames" нигде не присваивается значение, поэтому оно всегда будет иметь значение по умолчанию null.



In [47]:
// Задаём имена признаков (featureNames уже определён в ячейке 11)
DecisionTree.FeatureNames = featureNames;

var tree = new DecisionTree(maxDepth: 5);
tree.Fit(X_train, y_train);
int[] predTree = tree.Predict(X_test);

double accTree = Accuracy(y_test, predTree);
double f1Tree = MacroF1(y_test, predTree);
Console.WriteLine($"Decision Tree -> Accuracy: {accTree:F3}, Macro-F1: {f1Tree:F3}");

// Вывод правил
Console.WriteLine("\n--- Правила дерева ---");
tree.PrintTree();

// График важности признаков
var importance = tree.FeatureImportance();
var featNames = importance.Keys.ToArray();
var featValues = importance.Values.Select(v => (double)v).ToArray();

var barImportance = PlotlyCS.Column<string, double, string>(featNames, featValues)
    .WithTitle("Важность признаков (частота использования в дереве)")
    .WithYAxisStyle(Title.init("Количество разбиений"));
display(barImportance);

Decision Tree -> Accuracy: 0,966, Macro-F1: 0,967

--- Правила дерева ---
[PetalLength <= -0,76]
├─ Да:
│  ➤ класс 0
└─ Нет:
   [PetalLength <= 0,67]
   ├─ Да:
   │  [PetalWidth <= 0,59]
   │  ├─ Да:
   │  │  ➤ класс 1
   │  └─ Нет:
   │     [SepalWidth <= 0,11]
   │     ├─ Да:
   │     │  ➤ класс 2
   │     └─ Нет:
   │        ➤ класс 1
   └─ Нет:
      [PetalWidth <= 0,65]
      ├─ Да:
      │  [SepalLength <= 0,24]
      │  ├─ Да:
      │  │  [SepalWidth <= -1,39]
      │  │  ├─ Да:
      │  │  │  ➤ класс 2
      │  │  └─ Нет:
      │  │     ➤ класс 1
      │  └─ Нет:
      │     ➤ класс 2
      └─ Нет:
         ➤ класс 2


<!-- Plotly chart will be drawn inside this DIV -->

## 7. Важность признаков в дереве решений

In [48]:
// Убедимся, что переменные accLr, f1Lr, accSvm, f1Svm, accTree, f1Tree существуют
// Если какой-то нет, запустите соответствующие ячейки.

var modelNames = new[] { "Logistic Regression", "SVM", "Decision Tree" };
var accuracies = new[] { accLr, accSvm, accTree };
var f1s = new[] { f1Lr, f1Svm, f1Tree };

var accBar = PlotlyCS.Column<string, double, string>(modelNames, accuracies)
    .WithTitle("Сравнение точности (Accuracy)")
    .WithYAxisStyle(Title.init("Accuracy"));
display(accBar);

var f1Bar = PlotlyCS.Column<string, double, string>(modelNames, f1s)
    .WithTitle("Сравнение Macro F1")
    .WithYAxisStyle(Title.init("F1"));
display(f1Bar);

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

## 8. Вывод правил дерева решений (текстовая визуализация)

Правила уже выведены в ячейке 18. При необходимости можно повторно вызвать `tree.PrintTree()`.

## 9. Заключение

- Все три модели успешно обучены на масштабированных признаках Iris.
- Логистическая регрессия и SVM показали схожие высокие результаты.
- Дерево решений интерпретируемо: его правила и важность признаков наглядно демонстрируют, что длина и ширина лепестка являются главными разделяющими признаками.
- Визуализация правил дерева упрощает понимание модели.

Таким образом, цель лабораторной работы достигнута.